In [ ]:
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.sentence as nas
import nlpaug.flow as nafc
import numpy as np
import pandas as pd
import torch
import gc

from nlpaug.util import Action

rseed = 42

In [ ]:
# models
aug_syn_word = naw.SynonymAug(aug_src = "wordnet")

aug_syn_bert =naw.ContextualWordEmbsAug(
            model_path='bert-base-uncased',
            action="substitute"
        )

aug_backtranslation = aug = naw.BackTranslationAug(
        from_model_name="facebook/wmt19-en-de",
        to_model_name="facebook/wmt19-de-en"
    )

aug_rand_swap = naw.RandomWordAug(action = "swap")
aug_rand_del = naw.RandomWordAug(action = "delete")

In [3]:
# Synonym Augmenter
# prob 0.4
def synonymizer(text, aug1, aug2):
    coin = np.random.rand()
    if coin < 0.4:
        return aug1.augment(text)
    else:
        return aug2.augment(text)

In [4]:
# backtranslation
# prob 0.2
def back_translation(text, aug):
    return aug.augment(text)

# random swap
# prob 0.2
def random_swap(text, aug):
    return aug.augment(text)

#  random deletion
# prob 0.2
def random_del(text, aug):
    return aug.augment(text)

In [5]:
# def augmenter(text, aug_syn, aug_bt, aug_sw, aug_del):
#     coin = np.random.rand()
#     if coin < 0.4:
#         return synonymizer(text, aug_syn)
#     elif 0.4 < coin < 0.6:
#         return back_translation(text, aug_bt)
#     elif 0.6 < coin < 0.8:
#         return random_swap(text, aug_sw)
#     else:
#         return random_del(text, aug_del)




In [6]:
# data
df = pd.read_csv("..\data\csv\mbti_cleaned.csv")

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Tim\AppData\Local\Temp\ipykernel_12148\1371491495.py:2: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv("..\data\csv\mbti_cleaned.csv")


In [19]:
def label_samples(df, label, n_goal, rseed):
    obs_count = (df["label"] == label).sum()
    if obs_count < n_goal:
        df_subset = df[df["label"] == label]

        n_diff = n_goal - obs_count

        n_syn = round(n_diff*0.4)
        n_bt = round(n_diff*0.2)
        n_sw = round(n_diff*0.2)
        n_del = n_diff - (n_syn+n_bt+n_sw)



        syn_sample = df_subset.sample(n=n_syn, replace=True,random_state=rseed)
        bt_sample = df_subset.sample(n=n_bt, replace=True,random_state=rseed)
        sw_sample = df_subset.sample(n=n_sw, replace=True,random_state=rseed)
        del_sample = df_subset.sample(n=n_del, replace=True,random_state=rseed)

        return syn_sample, bt_sample, sw_sample, del_sample
    else:
        empty = df.iloc[:0]
        return empty, empty, empty, empty

    

    


In [24]:
syn_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
bt_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
sw_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
del_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])

for label in df["label"].unique():
    s1, s2, s3, s4 = label_samples(df, label, n_goal=10000, rseed=42)
    

    syn_sample = pd.concat([syn_sample, s1]).dropna()
    bt_sample = pd.concat([bt_sample, s2]).dropna()
    sw_sample = pd.concat([sw_sample, s3]).dropna()
    del_sample = pd.concat([del_sample, s4]).dropna()


    



In [ ]:
def data_augmenter(syn_sample, bt_sample, sw_sample, del_sample):
    #device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    
    aug_syn_word = naw.SynonymAug(aug_src = "wordnet")
    aug_syn_bert =naw.ContextualWordEmbsAug(
        model_path='bert-base-uncased',
        action="substitute",
        device="cuda"
    )

    syn_sample["post_augmented"] = syn_sample["post"].apply(synonymizer, args=(aug_syn_word, aug_syn_bert))
    # 1. Das nlpaug-Objekt löschen
    del aug_syn_word
    del aug_syn_bert

    # 2. Den Python Garbage Collector zwingen, die internen HF-Modelle freizugeben
    gc.collect()

    # 3. Den VRAM-Cache von PyTorch leeren
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    aug_backtranslation = naw.BackTranslationAug(
        from_model_name="facebook/wmt19-en-de",
        to_model_name="facebook/wmt19-de-en",
        device= "cuda"
    )

    bt_sample["post_augmented"] = bt_sample["post"].apply(back_translation, args=(aug_backtranslation))

    del aug_backtranslation
    gc.collect()

    # 3. Den VRAM-Cache von PyTorch leeren
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    aug_rand_swap = naw.RandomWordAug(action = "swap")
    sw_sample["post_augmented"] = sw_sample["post"].apply(random_swap, args = (aug_rand_swap))
    
    aug_rand_del = naw.RandomWordAug(action = "delete")
    del_sample["post_augmented"] = del_sample["post"].apply(random_del, args = (aug_rand_del))


    return syn_sample,bt_sample,sw_sample,del_sample

